# Bhojpuri Unicode WordPiece Tokenizer Training

## What is WordPiece?

**WordPiece** is a subword tokenization algorithm that handles rare words better than BPE.

### Why WordPiece for Bhojpuri?

Bhojpuri is a **very low-resource language** (only 3,234 lines of web data + HF corpus). WordPiece is better because:

| Aspect | Advantage for Bhojpuri |
|--------|------------------------|
| **Rare words** | Better breakdown of unknown words |
| **UNK rate** | Lower - fewer completely unknown tokens |
| **Robustness** | Works better with limited training data |
| **Likelihood** | Probability-based merging (smarter) |

**Key advantages:**
- ✅ Better OOV (out-of-vocabulary) handling
- ✅ Lower UNK rate
- ✅ Smarter rare word splitting
- ✅ **Perfect for low-resource languages** ⭐
- ✅ Probability-based (more linguistic)


## Setup & Seed Fixing

This notebook includes **SEED=42** for reproducible training.
Same seed = same tokenizer every run.


In [1]:
import json
import logging
import random
import string
import gc
import os
import tempfile
from datetime import datetime
from pathlib import Path
from collections import Counter

from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, decoders, trainers

# ============================================================================
# 🔧 SET SEED FOR REPRODUCIBILITY
# ============================================================================
SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
print(f"✓ Seed set to {SEED} for reproducibility")

# Try to import psutil
try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False

# ============================================================================
# CONFIGURATION
# ============================================================================
DATA_ROOT = Path("/kaggle/input/datasets/kspsvlnsiddardha/lma-slm/bhojpuri/data")

TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
TEST_DIR = DATA_ROOT / "test"
TOKENIZER_DIR = Path("/kaggle/working/")
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

LANG = "Bhojpuri"
LANG_SHORT = "bhojpuri"
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

print(f"✓ Data root: {DATA_ROOT}")
print(f"✓ Train dir exists: {TRAIN_DIR.exists()}")

✓ Seed set to 42 for reproducibility
✓ Data root: /kaggle/input/datasets/kspsvlnsiddardha/lma-slm/bhojpuri/data
✓ Train dir exists: True


In [2]:
# Utility functions
def gather_files(split_dir):
    return sorted(split_dir.glob("*.txt")) if split_dir.exists() else []

def total_bytes(files):
    return sum(f.stat().st_size for f in files)

def create_tokenizer():
    tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))
    tokenizer.normalizer = normalizers.NFC()
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    tokenizer.decoder = decoders.WordPiece(prefix="##")
    return tokenizer

def create_trainer(vocab_size):
    alphabet = list(string.printable)
    for i in range(0x0900, 0x0980):  # Devanagari
        alphabet.append(chr(i))
    return trainers.WordPieceTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
        initial_alphabet=alphabet,
        show_progress=True,
    )

# Discover files
train_files = gather_files(TRAIN_DIR)
val_files = gather_files(VAL_DIR)
test_files = gather_files(TEST_DIR)
train_val_files = train_files + val_files

# Vocab size
total = total_bytes(train_val_files)
est_tokens = total // 4
vocab_size = 16_000 if est_tokens < 200_000_000 else 32_000

print(f"\n📊 Corpus: {total / (1024**2):.1f} MB | Vocab: {vocab_size:,}")


📊 Corpus: 1001.4 MB | Vocab: 32,000


In [3]:
# Train tokenizer
logger.info("Creating WordPiece tokenizer...")
tokenizer = create_tokenizer()
trainer = create_trainer(vocab_size)

logger.info(f"Training on {len(train_val_files)} files...")
tokenizer.train(files=[str(f) for f in train_val_files], trainer=trainer)

vocab_actual = tokenizer.get_vocab_size()
logger.info(f"✓ Training complete. Vocab: {vocab_actual:,}")

[INFO] Creating WordPiece tokenizer...
[INFO] Training on 2 files...


[INFO] ✓ Training complete. Vocab: 32,000


In [4]:
# Save tokenizer
tokenizer_path = TOKENIZER_DIR / f"{LANG_SHORT}_wp_tokenizer.json"
tokenizer.save(str(tokenizer_path))

config = {
    "language": LANG,
    "tokenizer_type": "WordPiece",
    "vocab_size": vocab_actual,
    "seed": SEED,
    "created_at": datetime.now().isoformat(),
}

config_path = TOKENIZER_DIR / "wp_tokenizer_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"✓ Saved: {tokenizer_path.name}")
print(f"✓ Saved: {config_path.name}")

✓ Saved: bhojpuri_wp_tokenizer.json
✓ Saved: wp_tokenizer_config.json


In [5]:
# Evaluate on test set
token_freq = Counter()
total_tokens = 0
unk_count = 0

for test_file in test_files:
    with open(test_file) as f:
        for line in f:
            encoded = tokenizer.encode(line.strip())
            total_tokens += len(encoded.ids)
            token_freq.update(encoded.ids)
            unk_id = tokenizer.token_to_id("[UNK]")
            unk_count += sum(1 for tid in encoded.ids if tid == unk_id)

print(f"\n📊 Test Evaluation:")
print(f"  Vocab: {vocab_actual:,}")
print(f"  Total tokens: {total_tokens:,}")
print(f"  UNK rate: {100.0 * unk_count / total_tokens:.4f}%")
print(f"\n📈 Top tokens:")
for i, (tid, count) in enumerate(token_freq.most_common(10), 1):
    token = tokenizer.decode([tid])
    token_display = repr(token) if token == ' ' else token
    print(f"  {i}. {token_display:20s} - {count:6d}")


📊 Test Evaluation:
  Vocab: 32,000
  Total tokens: 11,391,403
  UNK rate: 0.0002%

📈 Top tokens:
  1. के                   - 762460
  2. ,                    - 346142
  3. ।                    - 266937
  4. में                  - 265234
  5. से                   - 175351
  6. आ                    - 142821
  7. -                    - 142456
  8. बा                   - 120054
  9. कि                   -  86465
  10. .                    -  74913


In [6]:
# ============================================================================
# Test on sample Bhojpuri text
# ============================================================================

print("\n" + "="*70)
print("TEST: Sample Bhojpuri Text Tokenization")
print("="*70)

test_samples = [
    "भोजपुरी भारत के बिहार में बोली जाती है।",
    "यह एक प्राचीन भाषा है।",
    "बहुत बढ़िया काम है यह।",
]

for i, text in enumerate(test_samples, 1):
    print(f"\n{i}. Input: {text}")
    encoded = tokenizer.encode(text)
    print(f"   Token IDs: {encoded.ids}")
    
    # Decode individual tokens
    tokens = [tokenizer.decode([tid]) for tid in encoded.ids]
    print(f"   Tokens: {tokens}")
    print(f"   Total: {len(encoded.ids)} tokens")


TEST: Sample Bhojpuri Text Tokenization

1. Input: भोजपुरी भारत के बिहार में बोली जाती है।
   Token IDs: [1002, 1046, 381, 1748, 385, 5364, 2520, 416, 218]
   Tokens: ['भोजपुरी', 'भारत', 'के', 'बिहार', 'में', 'बोली', 'जाती', 'है', '।']
   Total: 9 tokens

2. Input: यह एक प्राचीन भाषा है।
   Token IDs: [685, 470, 2699, 1143, 416, 218]
   Tokens: ['यह', 'एक', 'प्राचीन', 'भाषा', 'है', '।']
   Total: 6 tokens

3. Input: बहुत बढ़िया काम है यह।
   Token IDs: [669, 1391, 549, 416, 685, 218]
   Tokens: ['बहुत', 'बढ़िया', 'काम', 'है', 'यह', '।']
   Total: 6 tokens
